In [1]:
#1
import pandas as pd
import random
from datetime import date, timedelta
from faker import Faker

fake = Faker()
random.seed(42)  # keeps results consistent every time we run this

In [2]:
#2

# Each merchant has: canonical name, list of messy text variants, 
# starting price, frequency in days, and whether price increases over time
recurring_merchants = [
    {"name": "Netflix", "variants": ["NETFLIX.COM 8829 CA", "NETFLIX*STREAM", "NETFLIX.COM"], 
     "start_price": 199, "frequency_days": 30, "price_increases": [(6, 249), (14, 299)]},
    
    {"name": "Spotify", "variants": ["SPOTIFY P1A2B3", "SPOTIFY*PREMIUM", "SPOTIFY.COM"], 
     "start_price": 119, "frequency_days": 30, "price_increases": [(10, 149)]},
    
    {"name": "Amazon Prime", "variants": ["AMAZON PRIME MEM", "AMZN PRIME*MEMBER", "AMAZON.IN PRIME"], 
     "start_price": 1499, "frequency_days": 365, "price_increases": []},
    
    {"name": "Gym Membership", "variants": ["CULT.FIT MONTHLY", "CULTFIT*SUB", "CULT FIT"], 
     "start_price": 999, "frequency_days": 30, "price_increases": [(8, 1199)]},
    
    {"name": "Disney+ Hotstar", "variants": ["HOTSTAR SUB", "DISNEY+HOTSTAR", "HOTSTAR.COM"], 
     "start_price": 299, "frequency_days": 30, "price_increases": []},
    
    {"name": "Google One", "variants": ["GOOGLE*ONE STORAGE", "GOOGLE ONE 100GB", "GOOGLE ONE"], 
     "start_price": 130, "frequency_days": 30, "price_increases": []},
]

print(f"Defined {len(recurring_merchants)} recurring merchants")

Defined 6 recurring merchants


In [3]:
#3

def generate_recurring_transactions(merchant, months=18):
    transactions = []
    start_date = date.today() - timedelta(days=months * 30)
    current_date = start_date
    current_price = merchant["start_price"]
    month_counter = 0

    while current_date <= date.today():
        # Apply any scheduled price increase
        for increase_month, new_price in merchant["price_increases"]:
            if month_counter == increase_month:
                current_price = new_price

        # Pick a random messy variant of the merchant name (just like a real statement)
        variant = random.choice(merchant["variants"])

        # Add a little random jitter to the date (+/- 2 days) for realism
        jitter = random.randint(-2, 2)
        txn_date = current_date + timedelta(days=jitter)

        transactions.append({
            "date": txn_date,
            "raw_merchant_text": variant,
            "amount": current_price,
            "true_merchant_name": merchant["name"],  # answer key, not on real statements
            "is_recurring_ground_truth": True         # answer key, not on real statements
        })

        current_date += timedelta(days=merchant["frequency_days"])
        month_counter += 1

    return transactions

# Quick test on just Netflix
test_result = generate_recurring_transactions(recurring_merchants[0])
print(f"Generated {len(test_result)} Netflix transactions")
test_result[:3]

Generated 19 Netflix transactions


[{'date': datetime.date(2025, 2, 16),
  'raw_merchant_text': 'NETFLIX.COM',
  'amount': 199,
  'true_merchant_name': 'Netflix',
  'is_recurring_ground_truth': True},
 {'date': datetime.date(2025, 3, 20),
  'raw_merchant_text': 'NETFLIX.COM 8829 CA',
  'amount': 199,
  'true_merchant_name': 'Netflix',
  'is_recurring_ground_truth': True},
 {'date': datetime.date(2025, 4, 18),
  'raw_merchant_text': 'NETFLIX.COM 8829 CA',
  'amount': 199,
  'true_merchant_name': 'Netflix',
  'is_recurring_ground_truth': True}]

In [4]:
#4

def generate_oneoff_transactions(count=80):
    transactions = []
    for _ in range(count):
        txn_date = fake.date_between(start_date="-18M", end_date="today")
        merchant_text = fake.company().upper() + " " + fake.random_element(["PVT LTD", "STORE", "*PURCHASE", "RETAIL"])
        amount = round(random.uniform(50, 3000), 2)

        transactions.append({
            "date": txn_date,
            "raw_merchant_text": merchant_text,
            "amount": amount,
            "true_merchant_name": None,
            "is_recurring_ground_truth": False
        })
    return transactions

oneoff_result = generate_oneoff_transactions(10)
print(f"Generated {len(oneoff_result)} one-off transactions")
oneoff_result[:3]

Generated 10 one-off transactions


[{'date': datetime.date(2025, 8, 5),
  'raw_merchant_text': 'MURPHY, GARDNER AND LEE RETAIL',
  'amount': 323.6,
  'true_merchant_name': None,
  'is_recurring_ground_truth': False},
 {'date': datetime.date(2025, 2, 20),
  'raw_merchant_text': 'OCONNOR-PARKER *PURCHASE',
  'amount': 335.31,
  'true_merchant_name': None,
  'is_recurring_ground_truth': False},
 {'date': datetime.date(2025, 9, 24),
  'raw_merchant_text': 'PARKER-SIMMONS PVT LTD',
  'amount': 2550.11,
  'true_merchant_name': None,
  'is_recurring_ground_truth': False}]

In [5]:
#5

def generate_full_dataset():
    all_transactions = []

    # Add recurring transactions for every merchant we defined
    for merchant in recurring_merchants:
        all_transactions.extend(generate_recurring_transactions(merchant))

    # Add a batch of random one-off transactions
    all_transactions.extend(generate_oneoff_transactions(count=80))

    # Shuffle so it looks like a real, unordered bank statement
    random.shuffle(all_transactions)

    # Convert to a DataFrame and sort by date (like a real statement)
    df = pd.DataFrame(all_transactions)
    df = df.sort_values("date").reset_index(drop=True)
    return df

full_df = generate_full_dataset()
print(f"Total transactions: {len(full_df)}")
full_df.head(10)

Total transactions: 177


,date,raw_merchant_text,amount,true_merchant_name,is_recurring_ground_truth
0,2025-02-11,REED AND SONS *PURCHASE,325.13,NaN,False
1,2025-02-16,DISNEY+HOTSTAR,299.00,Disney+ Hotstar,True
2,2025-02-16,AMAZON.IN PRIME,1499.00,Amazon Prime,True
3,2025-02-18,SPOTIFY P1A2B3,119.00,Spotify,True
4,2025-02-18,GOOGLE*ONE STORAGE,130.00,Google One,True
5,2025-02-19,CULT FIT,999.00,Gym Membership,True
6,2025-02-20,NETFLIX.COM,199.00,Netflix,True
7,2025-03-01,GEORGE-YOUNG STORE,1730.54,NaN,False
8,2025-03-02,"HORTON, COOK AND BROWN PVT LTD",1636.02,NaN,False
9,2025-03-04,BRADFORD GROUP *PURCHASE,2273.02,NaN,False


In [6]:
#6
# The "realistic" version — what a real bank statement CSV would actually contain
bank_statement = full_df[["date", "raw_merchant_text", "amount"]]
bank_statement.to_csv("synthetic_bank_statement.csv", index=False)

# The "answer key" version — includes ground truth, used only for testing our pipeline
full_df.to_csv("synthetic_bank_statement_with_answers.csv", index=False)

print("Saved both CSV files successfully")
print(f"Files created in: {__import__('os').getcwd()}")

Saved both CSV files successfully
Files created in: C:\Users\DELL\Downloads\Projects\ghost-subscription-tracker


In [7]:
#Load the realistic CSV

df = pd.read_csv("synthetic_bank_statement.csv")
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} transactions")
df.head(10)

Loaded 177 transactions


,date,raw_merchant_text,amount
0,2025-02-11,REED AND SONS *PURCHASE,325.13
1,2025-02-16,DISNEY+HOTSTAR,299.00
2,2025-02-16,AMAZON.IN PRIME,1499.00
3,2025-02-18,SPOTIFY P1A2B3,119.00
4,2025-02-18,GOOGLE*ONE STORAGE,130.00
5,2025-02-19,CULT FIT,999.00
6,2025-02-20,NETFLIX.COM,199.00
7,2025-03-01,GEORGE-YOUNG STORE,1730.54
8,2025-03-02,"HORTON, COOK AND BROWN PVT LTD",1636.02
9,2025-03-04,BRADFORD GROUP *PURCHASE,2273.02


In [8]:
#Write the cleaning function

import re

def clean_merchant_text(raw_text):
    text = raw_text.upper()
    
    # Remove common transaction noise: trailing IDs, reference numbers, codes
    text = re.sub(r'\d{4,}', '', text)          # remove long number sequences (IDs/codes)
    text = re.sub(r'[*#]', ' ', text)             # replace special chars with space
    text = re.sub(r'\b(PVT LTD|LTD|LLC|INC|CO)\b', '', text)  # remove common company suffixes
    text = re.sub(r'\b(CA|IN|US|UK)\b', '', text)  # remove trailing country/state codes
    text = re.sub(r'\s+', ' ', text).strip()      # collapse multiple spaces
    
    return text

# Test it on a few examples
sample_texts = ["NETFLIX.COM 8829 CA", "NETFLIX*STREAM", "SPOTIFY P1A2B3", "CULT.FIT MONTHLY"]
for t in sample_texts:
    print(f"{t!r:35} -> {clean_merchant_text(t)!r}")

'NETFLIX.COM 8829 CA'               -> 'NETFLIX.COM'
'NETFLIX*STREAM'                    -> 'NETFLIX STREAM'
'SPOTIFY P1A2B3'                    -> 'SPOTIFY P1A2B3'
'CULT.FIT MONTHLY'                  -> 'CULT.FIT MONTHLY'


In [9]:
# Apply cleaning to the whole dataset
df["cleaned_text"] = df["raw_merchant_text"].apply(clean_merchant_text)
df[["raw_merchant_text", "cleaned_text"]].head(15)

,raw_merchant_text,cleaned_text
0,REED AND SONS *PURCHASE,REED AND SONS PURCHASE
1,DISNEY+HOTSTAR,DISNEY+HOTSTAR
2,AMAZON.IN PRIME,AMAZON. PRIME
3,SPOTIFY P1A2B3,SPOTIFY P1A2B3
4,GOOGLE*ONE STORAGE,GOOGLE ONE STORAGE
5,CULT FIT,CULT FIT
6,NETFLIX.COM,NETFLIX.COM
7,GEORGE-YOUNG STORE,GEORGE-YOUNG STORE
8,"HORTON, COOK AND BROWN PVT LTD","HORTON, COOK AND BROWN"
9,BRADFORD GROUP *PURCHASE,BRADFORD GROUP PURCHASE


In [10]:
#Fuzzy-match and cluster similar merchant names

from rapidfuzz import fuzz

def cluster_merchants(cleaned_texts, threshold=70):
    """
    Groups similar merchant text strings into clusters.
    Returns a dictionary mapping each unique cleaned text -> cluster ID
    """
    unique_texts = cleaned_texts.unique()
    clusters = {}       # cleaned_text -> cluster_id
    cluster_reps = []   # one representative string per cluster, for comparison

    for text in unique_texts:
        matched = False
        for i, rep in enumerate(cluster_reps):
            score = fuzz.token_sort_ratio(text, rep)
            if score >= threshold:
                clusters[text] = i
                matched = True
                break
        if not matched:
            cluster_reps.append(text)
            clusters[text] = len(cluster_reps) - 1

    return clusters, cluster_reps

clusters, cluster_reps = cluster_merchants(df["cleaned_text"], threshold=70)
df["cluster_id"] = df["cleaned_text"].map(clusters)

print(f"Found {len(cluster_reps)} unique merchant clusters from {df['cleaned_text'].nunique()} unique text variants")

Found 88 unique merchant clusters from 96 unique text variants


In [11]:
#Inspect the clusters

# Show which raw text variants ended up in the same cluster
cluster_summary = df.groupby("cluster_id")["cleaned_text"].unique()
for cid, texts in cluster_summary.items():
    if len(texts) > 1:  # only show clusters with more than one variant merged
        print(f"Cluster {cid}: {list(texts)}")

Cluster 4: ['GOOGLE ONE STORAGE', 'GOOGLE ONE']
Cluster 5: ['CULT FIT', 'CULTFIT SUB']
Cluster 11: ['YATES RETAIL', 'TERRY RETAIL']
Cluster 29: ['NORTON RETAIL', 'MARTIN RETAIL', 'POOLE RETAIL']
Cluster 32: ['POTTER, WILLIAMS AND BRANCH RETAIL', 'JORDAN, WILLIAMS AND HUNTER RETAIL']
Cluster 46: ['HALL-WILSON RETAIL', 'GILL-JOHNSON RETAIL']
Cluster 59: ['SOSA GROUP RETAIL', 'SOLIS GROUP RETAIL']


In [12]:
#12  raise the threshold and use a stricter scorer

def cluster_merchants_v2(cleaned_texts, threshold=85):
    unique_texts = cleaned_texts.unique()
    clusters = {}
    cluster_reps = []

    for text in unique_texts:
        matched = False
        for i, rep in enumerate(cluster_reps):
            # WRatio is more conservative than token_sort_ratio - 
            # it weighs full-string similarity more heavily, not just shared words
            score = fuzz.WRatio(text, rep)
            if score >= threshold:
                clusters[text] = i
                matched = True
                break
        if not matched:
            cluster_reps.append(text)
            clusters[text] = len(cluster_reps) - 1

    return clusters, cluster_reps

clusters, cluster_reps = cluster_merchants_v2(df["cleaned_text"], threshold=85)
df["cluster_id"] = df["cleaned_text"].map(clusters)

print(f"Found {len(cluster_reps)} unique merchant clusters from {df['cleaned_text'].nunique()} unique text variants")

cluster_summary = df.groupby("cluster_id")["cleaned_text"].unique()
for cid, texts in cluster_summary.items():
    if len(texts) > 1:
        print(f"Cluster {cid}: {list(texts)}")

Found 53 unique merchant clusters from 96 unique text variants
Cluster 0: ['REED AND SONS PURCHASE', 'REED RETAIL', 'MEDINA, BRADLEY AND DONOVAN RETAIL', 'POTTER, WILLIAMS AND BRANCH RETAIL', 'MORROW, JOHNSON AND THOMPSON STORE', 'NORRIS, CARRILLO AND LOPEZ RETAIL', 'RICHARDS, WHITE AND GONZALEZ RETAIL', 'JORDAN, WILLIAMS AND HUNTER RETAIL', 'HARRISON, HAYES AND COLLINS PURCHASE', 'WOODS, GRIMES AND SAUNDERS RETAIL', 'HERNANDEZ, HENDERSON AND WILLIAMS STORE', 'MOON, CHAMBERS AND PITTS PURCHASE', 'MILLER, CANTU AND VILLEGAS PURCHASE', 'WALKER, VASQUEZ AND BURGESS PURCHASE']
Cluster 4: ['GOOGLE ONE STORAGE', 'GOOGLE ONE']
Cluster 7: ['GEORGE-YOUNG STORE', 'HUNT, WHITE AND COOPER STORE', 'LEWIS, CARTER AND REID STORE', 'BECK, BENSON AND HANSEN STORE', 'RICHARDSON, KIRK AND BAUER STORE', 'NOLAN STORE']
Cluster 9: ['BRADFORD GROUP PURCHASE', 'HUBER GROUP', 'RIVERA PURCHASE']
Cluster 11: ['YATES RETAIL', 'ANDERSON PLC RETAIL', 'MILLER-WARD RETAIL', 'HARMON-ROACH RETAIL', 'BREWER GROUP RETAIL

In [13]:
#13  Strip generic suffix words, then re-cluster

def clean_merchant_text_v2(raw_text):
    text = raw_text.upper()
    
    text = re.sub(r'\d{4,}', '', text)
    text = re.sub(r'[*#.,]', ' ', text)
    text = re.sub(r'\b(PVT LTD|LTD|LLC|INC|CO|PLC)\b', '', text)
    text = re.sub(r'\b(CA|IN|US|UK)\b', '', text)
    
    # NEW: strip generic business words that cause false matches between
    # completely unrelated companies
    text = re.sub(r'\b(STORE|RETAIL|PURCHASE|GROUP|AND SONS|SUB|MEM|MEMBER)\b', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["cleaned_text_v2"] = df["raw_merchant_text"].apply(clean_merchant_text_v2)

def cluster_merchants_v3(cleaned_texts, threshold=88):
    unique_texts = cleaned_texts.unique()
    clusters = {}
    cluster_reps = []

    for text in unique_texts:
        matched = False
        for i, rep in enumerate(cluster_reps):
            # Plain ratio (not WRatio) - compares full strings directly,
            # won't be fooled by shared individual words
            score = fuzz.ratio(text, rep)
            if score >= threshold:
                clusters[text] = i
                matched = True
                break
        if not matched:
            cluster_reps.append(text)
            clusters[text] = len(cluster_reps) - 1

    return clusters, cluster_reps

clusters, cluster_reps = cluster_merchants_v3(df["cleaned_text_v2"], threshold=88)
df["cluster_id"] = df["cleaned_text_v2"].map(clusters)

print(f"Found {len(cluster_reps)} unique merchant clusters from {df['cleaned_text_v2'].nunique()} unique text variants")

cluster_summary = df.groupby("cluster_id")["cleaned_text_v2"].unique()
for cid, texts in cluster_summary.items():
    if len(texts) > 1:
        print(f"Cluster {cid}: {list(texts)}")

Found 93 unique merchant clusters from 95 unique text variants
Cluster 2: ['AMAZON PRIME', 'AMZN PRIME']
Cluster 5: ['CULT FIT', 'CULTFIT']


In [14]:
#Reattach the ground-truth columns for verification
# full_df and df have matching row order (df was loaded from a CSV saved directly from full_df)
df["true_merchant_name"] = full_df["true_merchant_name"].values
df["is_recurring_ground_truth"] = full_df["is_recurring_ground_truth"].values

print("Ground truth columns reattached")


Ground truth columns reattached


In [15]:
for merchant in recurring_merchants:
    merchant_rows = df[df["true_merchant_name"] == merchant["name"]]
    unique_clusters = merchant_rows["cluster_id"].nunique()
    cluster_ids = merchant_rows["cluster_id"].unique()
    status = "✅ CORRECT" if unique_clusters == 1 else "⚠️ SPLIT ACROSS CLUSTERS"
    print(f"{merchant['name']:20} -> {unique_clusters} cluster(s) {cluster_ids} {status}")

Netflix              -> 2 cluster(s) [ 6 14] ⚠️ SPLIT ACROSS CLUSTERS
Spotify              -> 3 cluster(s) [ 3 13 21] ⚠️ SPLIT ACROSS CLUSTERS
Amazon Prime         -> 1 cluster(s) [2] ✅ CORRECT
Gym Membership       -> 2 cluster(s) [ 5 31] ⚠️ SPLIT ACROSS CLUSTERS
Disney+ Hotstar      -> 3 cluster(s) [ 1 12 24] ⚠️ SPLIT ACROSS CLUSTERS
Google One           -> 3 cluster(s) [ 4 17 77] ⚠️ SPLIT ACROSS CLUSTERS


In [16]:
#Debug — see the actual cleaned strings per merchant

for merchant in recurring_merchants:
    variants = df[df["true_merchant_name"] == merchant["name"]]["cleaned_text_v2"].unique()
    print(f"{merchant['name']}: {list(variants)}")

Netflix: ['NETFLIX COM', 'NETFLIX STREAM']
Spotify: ['SPOTIFY P1A2B3', 'SPOTIFY PREMIUM', 'SPOTIFY COM']
Amazon Prime: ['AMAZON PRIME', 'AMZN PRIME']
Gym Membership: ['CULT FIT', 'CULTFIT', 'CULT FIT MONTHLY']
Disney+ Hotstar: ['DISNEY+HOTSTAR', 'HOTSTAR', 'HOTSTAR COM']
Google One: ['GOOGLE ONE STORAGE', 'GOOGLE ONE', 'GOOGLE ONE 100GB']


In [17]:
#Fix — switch to token_set_ratio, which compares shared words instead of raw characters
def cluster_merchants_v4(cleaned_texts, threshold=80):
    unique_texts = cleaned_texts.unique()
    clusters = {}
    cluster_reps = []

    for text in unique_texts:
        matched = False
        for i, rep in enumerate(cluster_reps):
            # token_set_ratio compares the SET of words in each string,
            # ignoring order and repeats - much better for "NETFLIX COM" vs "NETFLIX STREAM"
            score = fuzz.token_set_ratio(text, rep)
            if score >= threshold:
                clusters[text] = i
                matched = True
                break
        if not matched:
            cluster_reps.append(text)
            clusters[text] = len(cluster_reps) - 1

    return clusters, cluster_reps

clusters, cluster_reps = cluster_merchants_v4(df["cleaned_text_v2"], threshold=80)
df["cluster_id"] = df["cleaned_text_v2"].map(clusters)

print(f"Found {len(cluster_reps)} unique merchant clusters from {df['cleaned_text_v2'].nunique()} unique text variants")

# Re-run both checks
cluster_summary = df.groupby("cluster_id")["cleaned_text_v2"].unique()
print("\n--- Multi-variant clusters ---")
for cid, texts in cluster_summary.items():
    if len(texts) > 1:
        print(f"Cluster {cid}: {list(texts)}")

print("\n--- Recurring merchant verification ---")
for merchant in recurring_merchants:
    merchant_rows = df[df["true_merchant_name"] == merchant["name"]]
    unique_clusters = merchant_rows["cluster_id"].nunique()
    cluster_ids = merchant_rows["cluster_id"].unique()
    status = "\u2705 CORRECT" if unique_clusters == 1 else "\u26a0\ufe0f SPLIT ACROSS CLUSTERS"
    print(f"{merchant['name']:20} -> {unique_clusters} cluster(s) {cluster_ids} {status}")

Found 86 unique merchant clusters from 95 unique text variants

--- Multi-variant clusters ---
Cluster 2: ['AMAZON PRIME', 'AMZN PRIME']
Cluster 4: ['GOOGLE ONE STORAGE', 'GOOGLE ONE']
Cluster 5: ['CULT FIT', 'CULTFIT', 'CULT FIT MONTHLY']
Cluster 12: ['HOTSTAR', 'HOTSTAR COM']
Cluster 16: ['LEWIS CARTER AND REID', 'CARTER']
Cluster 29: ['POTTER WILLIAMS AND BRANCH', 'WILLIAMS']
Cluster 68: ['HERNANDEZ', 'HERNANDEZ HENDERSON AND WILLIAMS']
Cluster 84: ['COLE CAMPBELL AND WASHINGTON', 'CAMPBELL']

--- Recurring merchant verification ---
Netflix              -> 2 cluster(s) [ 6 14] ⚠️ SPLIT ACROSS CLUSTERS
Spotify              -> 3 cluster(s) [ 3 13 20] ⚠️ SPLIT ACROSS CLUSTERS
Amazon Prime         -> 1 cluster(s) [2] ✅ CORRECT
Gym Membership       -> 1 cluster(s) [5] ✅ CORRECT
Disney+ Hotstar      -> 2 cluster(s) [ 1 12] ⚠️ SPLIT ACROSS CLUSTERS
Google One           -> 2 cluster(s) [ 4 72] ⚠️ SPLIT ACROSS CLUSTERS


In [18]:
# Cluster on the primary (first) token instead of the whole string

def get_primary_token(text):
    words = text.split()
    return words[0] if words else text

df["primary_token"] = df["cleaned_text_v2"].apply(get_primary_token)

def cluster_merchants_v5(primary_tokens, threshold=88):
    unique_tokens = primary_tokens.unique()
    clusters = {}
    cluster_reps = []

    for token in unique_tokens:
        matched = False
        for i, rep in enumerate(cluster_reps):
            # partial_ratio checks if one string is a strong substring match of the other
            # this handles "CULT" vs "CULTFIT" and "HOTSTAR" vs "DISNEY+HOTSTAR"
            score = fuzz.partial_ratio(token, rep)
            if score >= threshold:
                clusters[token] = i
                matched = True
                break
        if not matched:
            cluster_reps.append(token)
            clusters[token] = len(cluster_reps) - 1

    return clusters, cluster_reps

clusters, cluster_reps = cluster_merchants_v5(df["primary_token"], threshold=88)
df["cluster_id"] = df["primary_token"].map(clusters)

print(f"Found {len(cluster_reps)} unique merchant clusters from {df['primary_token'].nunique()} unique primary tokens")

cluster_summary = df.groupby("cluster_id")["cleaned_text_v2"].unique()
print("\n--- Multi-variant clusters ---")
for cid, texts in cluster_summary.items():
    if len(texts) > 1:
        print(f"Cluster {cid}: {list(texts)}")

print("\n--- Recurring merchant verification ---")
for merchant in recurring_merchants:
    merchant_rows = df[df["true_merchant_name"] == merchant["name"]]
    unique_clusters = merchant_rows["cluster_id"].nunique()
    cluster_ids = merchant_rows["cluster_id"].unique()
    status = "\u2705 CORRECT" if unique_clusters == 1 else "\u26a0\ufe0f SPLIT ACROSS CLUSTERS"
    print(f"{merchant['name']:20} -> {unique_clusters} cluster(s) {cluster_ids} {status}")

Found 74 unique merchant clusters from 86 unique primary tokens

--- Multi-variant clusters ---
Cluster 1: ['DISNEY+HOTSTAR', 'HOTSTAR', 'HOTSTAR COM']
Cluster 3: ['SPOTIFY P1A2B3', 'SPOTIFY PREMIUM', 'SPOTIFY COM']
Cluster 4: ['GOOGLE ONE STORAGE', 'GOOGLE ONE', 'GOOGLE ONE 100GB']
Cluster 5: ['CULT FIT', 'CULTFIT', 'CULT FIT MONTHLY']
Cluster 6: ['NETFLIX COM', 'NETFLIX STREAM']
Cluster 8: ['HORTON COOK AND BROWN', 'NORTON']
Cluster 12: ['HUNT WHITE AND COOPER', 'PALMER-HUNT']
Cluster 14: ['MEDINA BRADLEY AND DONOVAN', 'MEDINA BENNETT AND PHILLIPS']
Cluster 22: ['BAKER', 'TATE-BAKER']
Cluster 24: ['POTTER WILLIAMS AND BRANCH', 'POTTS', 'HARPER-POTTER']
Cluster 28: ['RICHARDS-FISHER', 'RICHARDS WHITE AND GONZALEZ', 'RICHARDSON KIRK AND BAUER']
Cluster 30: ['MILLER-WARD', 'MILLER CANTU AND VILLEGAS']
Cluster 41: ['MARTIN', 'MARTINEZ-CHASE']
Cluster 43: ['VILLA-WILLIAMS', 'WILLIAMS']
Cluster 58: ['HERNANDEZ', 'HERNANDEZ HENDERSON AND WILLIAMS']

--- Recurring merchant verification ---
N

In [19]:
#Debug Amazon specifically

amazon_rows = df[df["true_merchant_name"] == "Amazon Prime"]
amazon_rows[["raw_merchant_text", "cleaned_text_v2", "primary_token", "cluster_id"]]

,raw_merchant_text,cleaned_text_v2,primary_token,cluster_id
2,AMAZON.IN PRIME,AMAZON PRIME,AMAZON,2
121,AMZN PRIME*MEMBER,AMZN PRIME,AMZN,54


In [20]:
# Add a small known-abbreviations dictionary before clustering
KNOWN_ABBREVIATIONS = {
    "AMZN": "AMAZON",
    "SBUX": "STARBUCKS",
    "MCD": "MCDONALDS",
}

def expand_abbreviations(token):
    return KNOWN_ABBREVIATIONS.get(token, token)

df["primary_token"] = df["primary_token"].apply(expand_abbreviations)

# Re-run clustering with the corrected primary tokens
clusters, cluster_reps = cluster_merchants_v5(df["primary_token"], threshold=88)
df["cluster_id"] = df["primary_token"].map(clusters)

print("--- Recurring merchant verification ---")
for merchant in recurring_merchants:
    merchant_rows = df[df["true_merchant_name"] == merchant["name"]]
    unique_clusters = merchant_rows["cluster_id"].nunique()
    cluster_ids = merchant_rows["cluster_id"].unique()
    status = "\u2705 CORRECT" if unique_clusters == 1 else "\u26a0\ufe0f SPLIT ACROSS CLUSTERS"
    print(f"{merchant['name']:20} -> {unique_clusters} cluster(s) {cluster_ids} {status}")

--- Recurring merchant verification ---
Netflix              -> 1 cluster(s) [6] ✅ CORRECT
Spotify              -> 1 cluster(s) [3] ✅ CORRECT
Amazon Prime         -> 1 cluster(s) [2] ✅ CORRECT
Gym Membership       -> 1 cluster(s) [5] ✅ CORRECT
Disney+ Hotstar      -> 1 cluster(s) [1] ✅ CORRECT
Google One           -> 1 cluster(s) [4] ✅ CORRECT


In [21]:
#Calculate gap and amount statistics per cluster

import numpy as np

def analyze_cluster(group):
    dates = sorted(group["date"])
    amounts = group["amount"].values
    
    if len(dates) < 2:
        gap_mean = None
        gap_std = None
    else:
        gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates) - 1)]
        gap_mean = np.mean(gaps)
        gap_std = np.std(gaps)
    
    return pd.Series({
        "transaction_count": len(group),
        "gap_mean_days": gap_mean,
        "gap_std_days": gap_std,
        "amount_mean": np.mean(amounts),
        "amount_std": np.std(amounts),
        "first_seen": min(dates),
        "last_seen": max(dates),
    })

cluster_stats = df.groupby("cluster_id").apply(analyze_cluster, include_groups=False).reset_index()
cluster_stats.head(10)

,cluster_id,transaction_count,gap_mean_days,gap_std_days,amount_mean,amount_std,first_seen,last_seen
0,0,2,35.000000,0.000000,890.830000,565.700000,2025-02-11,2025-03-18
1,1,19,30.055556,2.040485,299.000000,0.000000,2025-02-16,2026-08-11
2,2,2,368.000000,0.000000,1499.000000,0.000000,2025-02-16,2026-02-19
3,3,19,30.055556,1.580163,133.210526,14.979210,2025-02-18,2026-08-13
4,4,19,30.055556,1.544604,130.000000,0.000000,2025-02-18,2026-08-13
5,5,19,29.944444,2.040485,1114.789474,98.745595,2025-02-19,2026-08-12
6,6,19,29.833333,1.536591,246.368421,37.953171,2025-02-20,2026-08-11
7,7,1,NaN,NaN,1730.540000,0.000000,2025-03-01,2025-03-01
8,8,2,161.000000,0.000000,1806.855000,170.835000,2025-03-02,2025-08-10
9,9,1,NaN,NaN,2273.020000,0.000000,2025-03-04,2025-03-04


In [22]:
#Apply recurrence rules
def is_recurring(row):
    # Need at least 3 transactions to trust a pattern
    if row["transaction_count"] < 3:
        return False
    
    # Gap between charges should be consistent (low standard deviation)
    # A gap_std under ~5 days suggests a real monthly/weekly/annual rhythm
    if row["gap_std_days"] is None or row["gap_std_days"] > 5:
        return False
    
    # Gap should roughly resemble a real billing cycle: weekly, monthly, or annual
    plausible_cycle = any(abs(row["gap_mean_days"] - target) <= 5 for target in [7, 30, 365])
    if not plausible_cycle:
        return False
    
    return True

cluster_stats["is_recurring_predicted"] = cluster_stats.apply(is_recurring, axis=1)

recurring_clusters = cluster_stats[cluster_stats["is_recurring_predicted"] == True]
print(f"Flagged {len(recurring_clusters)} clusters as recurring, out of {len(cluster_stats)} total clusters")
recurring_clusters


Flagged 5 clusters as recurring, out of 73 total clusters


,cluster_id,transaction_count,gap_mean_days,gap_std_days,amount_mean,amount_std,first_seen,last_seen,is_recurring_predicted
1,1,19,30.055556,2.040485,299.000000,0.000000,2025-02-16,2026-08-11,True
3,3,19,30.055556,1.580163,133.210526,14.979210,2025-02-18,2026-08-13,True
4,4,19,30.055556,1.544604,130.000000,0.000000,2025-02-18,2026-08-13,True
5,5,19,29.944444,2.040485,1114.789474,98.745595,2025-02-19,2026-08-12,True
6,6,19,29.833333,1.536591,246.368421,37.953171,2025-02-20,2026-08-11,True


In [23]:
# Ground truth: which clusters actually contain a recurring merchant?
cluster_to_truth = df.groupby("cluster_id")["true_merchant_name"].apply(
    lambda x: x.notna().any() and x.iloc[0] is not None and (x == x.iloc[0]).all() and x.iloc[0] in [m["name"] for m in recurring_merchants]
)

cluster_stats["actually_recurring"] = cluster_stats["cluster_id"].map(cluster_to_truth)

tp = ((cluster_stats["is_recurring_predicted"] == True) & (cluster_stats["actually_recurring"] == True)).sum()
fp = ((cluster_stats["is_recurring_predicted"] == True) & (cluster_stats["actually_recurring"] == False)).sum()
fn = ((cluster_stats["is_recurring_predicted"] == False) & (cluster_stats["actually_recurring"] == True)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"True Positives: {tp}, False Positives: {fp}, False Negatives: {fn}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")

True Positives: 5, False Positives: 0, False Negatives: 1
Precision: 100.00%
Recall: 83.33%


In [24]:
# Build features for risk scoring
#For each recurring cluster, we calculate signals that suggest whether it's a good "cancel candidate":

def build_risk_features(row, cluster_id, df):
    cluster_txns = df[df["cluster_id"] == cluster_id].sort_values("date")
    
    # Days since the most recent charge (relative to "today" = last date in dataset)
    days_since_last = (df["date"].max() - row["last_seen"]).days
    
    # Price stability: coefficient of variation (std / mean) - lower = more stable price
    price_stability = row["amount_std"] / row["amount_mean"] if row["amount_mean"] > 0 else 0
    
    # Has the price increased over time? Compare first half vs second half of transactions
    half = len(cluster_txns) // 2
    if half > 0:
        early_avg = cluster_txns["amount"].iloc[:half].mean()
        late_avg = cluster_txns["amount"].iloc[half:].mean()
        price_increase_pct = ((late_avg - early_avg) / early_avg * 100) if early_avg > 0 else 0
    else:
        price_increase_pct = 0
    
    return pd.Series({
        "days_since_last_charge": days_since_last,
        "price_stability": price_stability,
        "price_increase_pct": price_increase_pct,
    })

recurring_clusters = cluster_stats[cluster_stats["is_recurring_predicted"] == True].copy()
risk_features = recurring_clusters.apply(lambda row: build_risk_features(row, row["cluster_id"], df), axis=1)
recurring_clusters = pd.concat([recurring_clusters, risk_features], axis=1)

recurring_clusters[["cluster_id", "amount_mean", "days_since_last_charge", "price_stability", "price_increase_pct"]]

,cluster_id,amount_mean,days_since_last_charge,price_stability,price_increase_pct
1,1,299.000000,2.0,0.000000,0.000000
3,3,133.210526,0.0,0.112448,22.689076
4,4,130.000000,0.0,0.000000,0.000000
5,5,1114.789474,1.0,0.088578,17.408334
6,6,246.368421,2.0,0.154050,27.047913


In [25]:
#Convert features into an explainable risk score

def calculate_risk_score(row):
    score = 0
    reasons = []
    
    # Signal 1: price has increased significantly - classic "silent price creep"
    if row["price_increase_pct"] > 10:
        score += 0.35
        reasons.append(f"price increased {row['price_increase_pct']:.0f}% over time")
    
    # Signal 2: very stable price (never questioned/renegotiated) - often means "on autopilot"
    if row["price_stability"] < 0.05:
        score += 0.25
        reasons.append("price has never changed (classic 'set and forget' pattern)")
    
    # Signal 3: hasn't charged recently relative to dataset - could mean cancelled already, skip
    if row["days_since_last_charge"] > 45:
        score += 0.20
        reasons.append("no recent charge detected")
    
    # Base signal: every confirmed recurring subscription gets a baseline risk
    score += 0.20
    reasons.append("confirmed recurring pattern")
    
    score = min(score, 1.0)
    return pd.Series({"risk_score": round(score, 2), "risk_reasons": "; ".join(reasons)})

risk_results = recurring_clusters.apply(calculate_risk_score, axis=1)
recurring_clusters = pd.concat([recurring_clusters, risk_results], axis=1)

recurring_clusters[["cluster_id", "amount_mean", "price_increase_pct", "risk_score", "risk_reasons"]].sort_values("risk_score", ascending=False)


,cluster_id,amount_mean,price_increase_pct,risk_score,risk_reasons
3,3,133.210526,22.689076,0.55,price increased 23% over time; confirmed recur...
6,6,246.368421,27.047913,0.55,price increased 27% over time; confirmed recur...
5,5,1114.789474,17.408334,0.55,price increased 17% over time; confirmed recur...
1,1,299.000000,0.000000,0.45,price has never changed (classic 'set and forg...
4,4,130.000000,0.000000,0.45,price has never changed (classic 'set and forg...


In [26]:
# The "what if I cancelled" savings simulator
#This takes your flagged subscriptions and calculates projected annual 
#savings — the interactive centerpiece feature from your presentation\

def calculate_annual_cost(row):
    # Estimate how many times per year this subscription charges, based on its gap pattern
    if row["gap_mean_days"] <= 10:
        charges_per_year = 52  # weekly
    elif row["gap_mean_days"] <= 45:
        charges_per_year = 12  # monthly
    else:
        charges_per_year = 1   # annual
    
    return row["amount_mean"] * charges_per_year

recurring_clusters["annual_cost"] = recurring_clusters.apply(calculate_annual_cost, axis=1)

# Pull in a readable merchant name for display (using the most common cleaned text per cluster)
def get_display_name(cluster_id):
    names = df[df["cluster_id"] == cluster_id]["cleaned_text_v2"]
    return names.mode()[0] if not names.empty else f"Cluster {cluster_id}"

recurring_clusters["merchant_display_name"] = recurring_clusters["cluster_id"].apply(get_display_name)

summary = recurring_clusters[["merchant_display_name", "amount_mean", "annual_cost", "risk_score", "risk_reasons"]].sort_values("risk_score", ascending=False)
summary.columns = ["Subscription", "Monthly/Charge Amount", "Est. Annual Cost", "Risk Score", "Why Flagged"]
summary

,Subscription,Monthly/Charge Amount,Est. Annual Cost,Risk Score,Why Flagged
3,SPOTIFY P1A2B3,133.210526,1598.526316,0.55,price increased 23% over time; confirmed recur...
6,NETFLIX COM,246.368421,2956.421053,0.55,price increased 27% over time; confirmed recur...
5,CULT FIT,1114.789474,13377.473684,0.55,price increased 17% over time; confirmed recur...
1,HOTSTAR,299.000000,3588.000000,0.45,price has never changed (classic 'set and forg...
4,GOOGLE ONE STORAGE,130.000000,1560.000000,0.45,price has never changed (classic 'set and forg...


In [27]:
#The actual simulator — pick subscriptions to "cancel" and see savings

def simulate_savings(cancel_list, clusters_df):
    """
    cancel_list: list of merchant_display_name strings the user wants to hypothetically cancel
    """
    to_cancel = clusters_df[clusters_df["merchant_display_name"].isin(cancel_list)]
    total_savings = to_cancel["annual_cost"].sum()
    
    print(f"If you cancelled: {', '.join(cancel_list)}")
    print(f"You would save approximately \u20b9{total_savings:,.2f} per year")
    return total_savings

# Example: simulate cancelling the two highest-risk subscriptions
top_two = summary.head(2)["Subscription"].tolist()
simulate_savings(top_two, recurring_clusters)

If you cancelled: SPOTIFY P1A2B3, NETFLIX COM
You would save approximately ₹4,554.95 per year


np.float64(4554.947368421053)